<a href="https://colab.research.google.com/github/vikamaslancuk5-commits/AB_Testing_Statistical_Significance_Analysis/blob/main/AB_Testing_Statistical_Significance_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[Дашборд АВ тестування](https://public.tableau.com/app/profile/viktoriia.maslianchuk/viz/ABTest_17843686579510/ABTest)

[Результати](https://drive.google.com/file/d/1VuNEZ4wAwnweW2aE5rbZvhjAtrysHi4J/view?usp=sharing)

[Дашборд оцінки статистичної значущості A/B-тестів](https://public.tableau.com/app/profile/viktoriia.maslianchuk/viz/ABTestingSignificance/Dashboard1?publish=yes)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest


file_path = '/content/drive/MyDrive/Mate/AB-test.csv'
df = pd.read_csv(file_path)

# Очищаємо назви подій
df['event_name_clean'] = df['event_name'].astype(str).str.strip().str.lower()


metrics = {
    'add_payment_info / session':  ('add_payment_info', 'session', 'add_payment_info'),
    'add_shipping_info / session': ('add_shipping_info', 'session', 'add_shipping_info'),
    'begin_checkout / session':    ('begin_checkout',    'session', 'begin_checkout'),
    'new_accounts / session':      ('new accounts',      'session', 'new account')
}


results = []

# 4. Отримуємо ВСІ унікальні тести з файлу (1, 2, 3, 4...)
target_tests = sorted(df['test'].unique())

for test_number in target_tests:
    test_data = df[df['test'] == test_number]

    ctr_data = test_data[test_data['test_group'] == 1]
    var_data = test_data[test_data['test_group'] == 2]

    for metric_name, (num_display, den_display, num_search) in metrics.items():
        den_search = 'sesion'

        #  (Test / Group 2)
        denominator_count_test = var_data[var_data['event_name_clean'] == den_search]['value'].sum()
        numerator_count_test = var_data[var_data['event_name_clean'] == num_search]['value'].sum()
        conversion_rate_test = numerator_count_test / denominator_count_test if denominator_count_test > 0 else np.nan

        #  КОНТРОЛЬНА ГРУПА (Control / Group 1)
        denominator_count_control = ctr_data[ctr_data['event_name_clean'] == den_search]['value'].sum()
        numerator_count_control = ctr_data[ctr_data['event_name_clean'] == num_search]['value'].sum()
        conversion_rate_control = numerator_count_control / denominator_count_control if denominator_count_control > 0 else np.nan

        #  ВІДНОСНА ЗМІНА МЕТРИКИ (%)
        metric_change = (conversion_rate_test - conversion_rate_control) / conversion_rate_control * 100 if conversion_rate_control and conversion_rate_control > 0 else np.nan

        #  Z-ТЕСТ
        if denominator_count_test > 0 and denominator_count_control > 0:
            z_stat, p_value = proportions_ztest(
                count=[numerator_count_test, numerator_count_control],
                nobs=[denominator_count_test, denominator_count_control]
            )
            is_significant = "TRUE" if p_value < 0.05 else "FALSE"
        else:
            z_stat, p_value, is_significant = np.nan, np.nan, "FALSE"

        results.append({
            "test_number": test_number,
            "metric": metric_name,
            "numerator_event": num_display,
            "denominator_event": den_display,
            "numerator_count_test": numerator_count_test,
            "denominator_count_test": denominator_count_test,
            "conversion_rate_test": conversion_rate_test,
            "numerator_count_control": numerator_count_control,
            "denominator_count_control": denominator_count_control,
            "conversion_rate_control": conversion_rate_control,
            "metric_change_%": metric_change,
            "z_stat": z_stat,
            "p_value": p_value,
            "significant": is_significant
        })

#  Формування та збереження DataFrame
df_results = pd.DataFrame(results)
df_results.to_csv('results.csv', index=False, sep=',', encoding='utf-8')
output_path = '/content/drive/MyDrive/Mate/results.csv'
df_results.to_csv(output_path, index=False, encoding='utf-8-sig')

df_results.head()

,test_number,metric,numerator_event,denominator_event,numerator_count_test,denominator_count_test,conversion_rate_test,numerator_count_control,denominator_count_control,conversion_rate_control,metric_change_%,z_stat,p_value,significant
0,1,add_payment_info / session,add_payment_info,session,2229,45193,0.049322,1988,45362,0.043825,12.542021,3.924884,0.000087,TRUE
1,1,add_shipping_info / session,add_shipping_info,session,3221,45193,0.071272,3034,45362,0.066884,6.560481,2.603571,0.009226,TRUE
2,1,begin_checkout / session,begin_checkout,session,4021,45193,0.088974,3784,45362,0.083418,6.660587,2.978783,0.002894,TRUE
3,1,new_accounts / session,new accounts,session,3681,45193,0.081451,3823,45362,0.084278,-3.354299,-1.542883,0.122859,FALSE
4,2,add_payment_info / session,add_payment_info,session,2409,50244,0.047946,2344,50637,0.046290,3.576911,1.240994,0.214608,FALSE
